In [1]:
# DS4400 HW 4
# Eunchae Hong
# Problem 3: Naive Bayes Classifier

import numpy as np
import pandas as pd
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)
import matplotlib.pyplot as plt

# load the mushroom database in
mushroom_data = pd.read_csv("mushroom/agaricus-lepiota.data", header = None)

In [2]:
'''
    Part 1
    Train the Naive Bayes classifier.
    Compute the prior probabilities for the Edible and Poisonous classes from the training data.
    For each feature X_i in the dataset compute the probabilities P[X_i = x| Y={Edible}], and P[X_i = x| Y={Poisonous}]$ from the training data.
    Use the Laplace smoothing method when computing these probabilities.
    Note that the Naive Bayes classifier stores these prior and conditional probabilities.
'''
# implment the train test split
X = mushroom_data.iloc[:, 1:]
y = mushroom_data.iloc[:, 0]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 5, stratify = y)

# create a training function
def naive_bayes_train(X_train, y_train):
    classes = y_train.unique()
    priors = {}
    likelihoods = {}

    # compute prior probabilities
    for c in classes:
        priors[c] = np.mean(y_train == c)

    # compute conditional probabilities with Laplace smoothing
    for c in classes:
        likelihoods[c] = {}
        X_train_c = X_train[y_train == c]
        n_c = len(X_train_c)

        for col in X_train.columns:
            likelihoods[c][col] = {}
            value_counts = X_train_c[col].value_counts()
            n_vals = X_train[col].nunique()

            for val in X_train[col].unique():
                count = value_counts.get(val, 0)
                likelihoods[c][col][val] = (count + 1) / (n_c + n_vals)
    
    return priors, likelihoods

# create a predicition function
def naive_bayes_predict(X_test, priors, likelihoods):
    classes = list(priors.keys())
    predictions = []

    for _, row in X_test.iterrows():
        class_scores = {}

        for c in classes:
            score = np.log(priors[c])
            for col in X_test.columns:
                val = row[col]
                prob = likelihoods[c][col].get(val, 1e-6)
                score += np.log(prob)
            class_scores[c] = score
        predictions.append(max(class_scores, key = class_scores.get))
    
    return predictions

# implment the training and predicition functions
priors, likelihoods = naive_bayes_train(X_train, y_train)
y_pred = naive_bayes_predict(X_test, priors, likelihoods)

# print the prior probabilities
print("Prior Probabilities:")
for c, p in priors.items():
    print(f"  P(Y = {c}) = {round(p, 3)}")

Prior Probabilities:
  P(Y = e) = 0.518
  P(Y = p) = 0.482


In [8]:
'''
    Part 2
    For each point in the testing set estimate the probability that it belongs to the Edible and Poisonous classes. 
    Use the Naive Bayes classifier probabilities computed in part (1).
'''

# create a function that predicts the probabilties
def naive_bayes_predict_probs(X_test, priors, likelihoods):
    classes = list(priors.keys())
    all_probs = []

    for _, row in X_test.iterrows():
        class_scores = {}
        for c in classes:
            score = np.log(priors[c])
            for col in X_test.columns:
                val = row[col]
                prob = likelihoods[c][col].get(val, 1e-6)
                score += np.log(prob)
            class_scores[c] = score

        max_score = max(class_scores.values())
        exp_scores = {c: np.exp(class_scores[c] - max_score) for c in classes}
        total = sum(exp_scores.values())
        probs = {c: exp_scores[c] / total for c in classes}
        all_probs.append(probs)

    return pd.DataFrame(all_probs)

# get the probablity estimates at each test point
prob_df = naive_bayes_predict_probs(X_test, priors, likelihoods)
prob_df.columns = ["Edible", "Poisonous"]
print(prob_df)

            Edible     Poisonous
0     1.000000e+00  4.493912e-11
1     1.000000e+00  2.155573e-08
2     1.487877e-11  1.000000e+00
3     9.999999e-01  5.908568e-08
4     5.583130e-10  1.000000e+00
...            ...           ...
2026  1.000000e+00  8.300295e-11
2027  2.062547e-04  9.997937e-01
2028  4.068350e-15  1.000000e+00
2029  1.185592e-11  1.000000e+00
2030  9.999981e-01  1.920159e-06

[2031 rows x 2 columns]


In [17]:
'''
    Part 3
    Compute accuracy, precision, recall, and F1 score for your Naive Bayes classifier on the testing data.
'''

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test,y_pred, pos_label = "p")
recall = recall_score(y_test, y_pred, pos_label = "p")
f1 = f1_score(y_test, y_pred, pos_label = "p")

print("Naive Bayes Test Metrics")
print("-" * 20)
print(f"Accuracy: {round(accuracy, 3)}\nPrecision: {round(precision, 3)}")
print(f"Recall: {round(recall, 3)}\nF1 Score: {round(f1, 3)}")

Naive Bayes Test Metrics
--------------------
Accuracy: 0.956
Precision: 1.0
Recall: 0.909
F1 Score: 0.952


In [18]:
'''
    Part 4
    Compare the results obtained by your implementation with those obtained with a Naive Bayes package (trained on the same dataset).
    Use several metrics, including accuracy, precision, recall, and F1 score. Are the results similar or different?
'''

# need to encode the features to use CategoricalNB
encoder = OrdinalEncoder()
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

# train the package model and use Laplace smoothing
naive_bayes_pack = CategoricalNB(alpha = 1)
naive_bayes_pack.fit(X_train_encoded, y_train)

# predict probabilities
y_pred_pack = naive_bayes_pack.predict(X_test_encoded)

# calculate the metrics
pack_accuracy = accuracy_score(y_test, y_pred_pack)
pack_precision = precision_score(y_test, y_pred_pack, pos_label = "p")
pack_recall = recall_score(y_test, y_pred_pack, pos_label = "p")
pack_f1 = f1_score(y_test, y_pred_pack, pos_label = "p")

# print metrics
print("Naive Bayes Test Metrics with Package")
print("-" * 20)
print(f"Accuracy: {round(pack_accuracy, 3)}\nPrecision: {round(pack_precision, 3)}")
print(f"Recall: {round(pack_recall, 3)}\nF1 Score: {round(pack_f1, 3)}")

Naive Bayes Test Metrics with Package
--------------------
Accuracy: 0.956
Precision: 1.0
Recall: 0.909
F1 Score: 0.952


***Part 4 Questions: Use several metrics, including accuracy, precision, recall, and F1 score. Are the results similar or different?***

After calculating the accuracy, precision, recall, and F1 scores for both the model I have implmented and the model from the package, all the metrics are the same (Accuracy: 0.956, Precision: 1.0, Recall, 0.909, and F1 Score: 0.952).